<span style="color:red">**===================================================================================**</span>

<span style="color:red">**===================================================================================**</span>

<span style="color:red">**####################---- SETUP DATASET AND INSTALL REQUIRED LIBRARIES ----####################**</span>



**<span style="color:blue">--- Move Dataset and Code folders to Kaggle's working place. </span>**

In [ ]:
#RUN__1
# Copy folders from /kaggle/input to /kaggle/working
!cp -r /kaggle/input/yolov-dp/YOLO-DP-main /kaggle/working/

# List all files/folders in the working directory to verify the copy
print("✅ List working folder:")
!ls /kaggle/working/YOLO-DP-main

**<span style="color:blue">--- Install required libraries from Requirement.txt file. </span>**

In [ ]:
#RUN__2
#Install Requirement.txt
!pip install --quiet -r /kaggle/working/YOLO-DP-main/yolov-DP/requirements.txt --use-deprecated=legacy-resolver

import torch, torchvision, cv2, PIL, numpy, pandas, yaml, tqdm, matplotlib, scipy, tensorboard

print("✅ torch:", torch.__version__)
print("✅ torchvision:", torchvision.__version__)
print("✅ opencv-python:", cv2.__version__)
print("✅ Pillow:", PIL.__version__)
print("✅ numpy:", numpy.__version__)
print("✅ pandas:", pandas.__version__)
print("✅ PyYAML:", yaml.__version__)
print("✅ tqdm:", tqdm.__version__)
print("✅ matplotlib:", matplotlib.__version__)
print("✅ scipy:", scipy.__version__)
print("✅ tensorboard:", tensorboard.__version__)

**<span style="color:blue">--- Install lost Library. </span>**

In [ ]:
#RUN__3
#Install Lost Library

!pip install efficientnet-pytorch
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0+cu118 -f https://download.pytorch.org/whl/torch_stable.html
!pip install protobuf==3.20.* tensorboard==2.12

####
!pip install pillow==9.5.0 --force-reinstall


<span style="color:red">**===================================================================================**</span>

<span style="color:red">**===================================================================================**</span>

<span style="color:red">**####################---- PDT_DATASET TRAINING ----####################**</span>

NOTE: This dataset contains crop images annotated for a single class “unhealthy”, aiming to detect and localize unhealthy regions of plants.

**<span style="color:blue">--- Prepare pdt_data.yaml file. </span>**

In [ ]:
%%writefile pdt_data.yaml

train: /kaggle/working/YOLO-DP-main/PDT_dataset/LL/YOLO_txt/train/images
val: /kaggle/working/YOLO-DP-main/PDT_dataset/LL/YOLO_txt/val/images 
test: /kaggle/working/YOLO-DP-main/PDT_dataset/LL/YOLO_txt/test/images 

nc: 1  
names: ['unhealthy']

In [ ]:
#Print pdt_data.yaml
!cat /kaggle/working/pdt_data.yaml

**<span style="color:blue">--- Prepare model file. </span>**

In [ ]:
%%writefile yolov5-DP-Fast.yaml

nc: 80
depth_multiple: 0.33
width_multiple: 0.50
anchors:
  - [10,13, 16,30, 33,23]
  - [30,61, 62,45, 59,119]
  - [116,90, 156,198, 373,326]

backbone:
  [[-1, 1, Conv, [64, 6, 2, 2]],
   [-1, 1, Conv, [128, 3, 2]],
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],
   [-1, 6, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],
   [-1, 3, C3, [1024]],
   [-1, 1, SPPF, [1024, 5]],
  ]

head:
  [[-1, 1, GhostConv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],
   [-1, 3, C3, [512, False]],
   [-1, 1, GhostConv, [512, 3, 2]],
   [[-1, 10], 1, Concat, [1]],
   [-1, 3, C3, [1024, False]],
   [[13, 16, 16], 1, Detect, [nc, anchors]],
  ]


In [ ]:
#Print model
!cat /kaggle/working/yolov5-DP-Fast.yaml

**<span style="color:blue">--- Start training pdt_data.yaml. </span>**

In [ ]:
#RUN__1
!python /kaggle/working/YOLO-DP-main/yolov-DP/train.py \
  --img 640 \
  --batch 16 \
  --epochs 50 \
  --data /kaggle/working/pdt_data.yaml \
  --cfg /kaggle/working/yolov5-DP-Fast.yaml \
  --weights '' \
  --device 0 \
  --name yolov5s-dense

**<span style="color:blue">--- Start evaluate training Model YOLOv-DP on PDT Dataset. </span>**

In [ ]:
# === 🔧 Patch val.py to use correct weights path ===

import re

val_file = "/kaggle/working/YOLO-DP-main/yolov-DP/val.py"

# Đọc file gốc
with open(val_file, "r") as f:
    content = f.read()

# Thay đường dẫn mặc định của --weights
content = re.sub(
    r"default=.*fanjiaying.*best\.pt['\"]",
    "default='/kaggle/working/YOLO-DP-main/yolov-DP/runs/train/yolov5s-dense3/weights/best.pt'",
    content
)

# Ghi lại file
with open(val_file, "w") as f:
    f.write(content)

print("✅ Patched val.py successfully! Now default weights path points to your Kaggle model.")


In [ ]:
#RUN__2
!python /kaggle/working/YOLO-DP-main/yolov-DP/val.py \
    --weights /kaggle/working/YOLO-DP-main/yolov-DP/runs/train/yolov5s-dense/weights/best.pt \
    --data /kaggle/working/pdt_data.yaml \
    --img 640 \
    --batch 16 \
    --device 0 \
    > /kaggle/working/val_log.txt 2>&1

In [ ]:
#RUN__3
import os
import matplotlib.pyplot as plt
import cv2

# Directory containing results (change if needed)
results_dir = "/kaggle/working/YOLO-DP-main/yolov-DP/runs/val/exp"

# Get list of image files
images = [f for f in os.listdir(results_dir) if f.endswith((".png", ".jpg"))]

# Number of columns (2 vertical columns)
cols = 2
rows = (len(images) + cols - 1) // cols  # auto rows depending on number of images

plt.figure(figsize=(12, rows * 5))  # Adjust figure size

for i, img_file in enumerate(images):
    img_path = os.path.join(results_dir, img_file)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot in a grid (2 columns)
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(img_file, fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import re
import pandas as pd

# Read log file
with open("/kaggle/working/val_log.txt", "r") as f:
    log = f.read()

# Extract metrics using regex
precision = float(re.search(r"all\s+\d+\s+\d+\s+([\d.]+)", log).group(1))
recall    = float(re.search(r"all\s+\d+\s+\d+\s+[\d.]+\s+([\d.]+)", log).group(1))
map50     = float(re.search(r"all\s+\d+\s+\d+\s+[\d.]+\s+[\d.]+\s+([\d.]+)", log).group(1))
map5095   = float(re.search(r"all\s+\d+\s+\d+\s+[\d.]+\s+[\d.]+\s+[\d.]+\s+([\d.]+)", log).group(1))

# Compute F1
f1 = 2 * (precision * recall) / (precision + recall + 1e-16)

# Extract GFLOPs and Params
gflops = re.search(r"([\d.]+)\s+GFLOPs", log).group(1)
params = re.search(r"(\d+)\s+parameters", log).group(1)

# Extract FPS (from Speed log)
speed_match = re.search(r"Speed: .*?([\d.]+)ms pre-process, ([\d.]+)ms inference, ([\d.]+)ms NMS", log)
if speed_match:
    total_ms = sum(float(x) for x in speed_match.groups())
    fps = round(1000 / total_ms, 2)
else:
    fps = "N/A"

summary_dict = {
    "Precision (P)": precision,
    "Recall (R)": recall,
    "mAP@0.5": map50,
    "mAP@0.5:0.95": map5095,
    "F1 Score": f1,
    "GFLOPs": gflops,
    "Parameters": params,
    "FPS": fps,
    "Pre-trained": "No"  # because you trained from scratch
}

# Convert to 2-column DataFrame
summary_df = pd.DataFrame(list(summary_dict.items()), columns=["Metric", "Value"])

print("📊 Final Evaluation Metrics:")
display(summary_df)

**<span style="color:blue">--- Start detect PDT_dataset/YOLO_txt/test/images folder training Model YOLOv-DP on CWC Dataset. </span>**

In [ ]:
#RUN__4
!python /kaggle/working/YOLO-DP-main/yolov-DP/detect.py \
  --weights /kaggle/working/YOLO-DP-main/yolov-DP/runs/train/yolov5s-dense/weights/best.pt \
  --img 1024 \
  --conf 0.25 \
  --source /kaggle/working/YOLO-DP-main/PDT_dataset/LH/images \
  --name detect_results

In [ ]:
#RUN__5
import os
import glob
import cv2
import matplotlib.pyplot as plt

results_path = "/kaggle/working/YOLO-DP-main/yolov-DP/runs/detect/detect_results"
# lấy tất cả ảnh có phần mở rộng .jpg, .JPG, .png
images = []
for ext in ["*.jpg", "*.JPG", "*.png", "*.PNG"]:
    images.extend(glob.glob(os.path.join(results_path, ext)))

images = sorted(images)

print(f"Found {len(images)} images")

cols = 2
rows = (len(images) + cols - 1) // cols

plt.figure(figsize=(12, rows * 6))
for idx, img_path in enumerate(images):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(rows, cols, idx + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(os.path.basename(img_path), fontsize=8)

plt.tight_layout()
plt.show()


<span style="color:red">**===================================================================================**</span>

<span style="color:red">**===================================================================================**</span>

<span style="color:red">**####################---- DONE AND ANNOUCE ----####################**</span>

In [ ]:
#COMPRESS TO DOWNLOAD
import shutil

# Compress the entire /kaggle/working/ directory into a ZIP file
# Output will be saved as /kaggle/working/yolovDP_TRAINED.zip
shutil.make_archive("/kaggle/working/yolovDP_100_TRAINED", 'zip', "/kaggle/working/YOLO-DP-main/yolov-DP")

print("✅ Compression completed. File saved at /kaggle/working/working_dir.zip")

In [ ]:
#CODE SEND GMAIL TO ANNOUCE 
import smtplib
from email.mime.text import MIMEText

# ==== CONFIGURATION ====
gmail_user = "papervhk2003@gmail.com"
gmail_password = "gcwlokbgzgteimjy"   # App password (remove spaces)
to_email = "khg03jobs@gmail.com"  # Recipient email (can be your own)

subject = "Kaggle Training Completed ✅"
body = "Hello Khang, your PROJECT training process has finished!"

# ==== CREATE EMAIL CONTENT ====
msg = MIMEText(body)
msg["Subject"] = subject
msg["From"] = gmail_user
msg["To"] = to_email

# ==== SEND EMAIL ====
try:
    server = smtplib.SMTP_SSL("smtp.gmail.com", 465)
    server.login(gmail_user, gmail_password)
    server.sendmail(gmail_user, [to_email], msg.as_string())
    server.quit()
    print("📩 Email sent successfully!")
except Exception as e:
    print("❌ Error while sending email:", e)